# Step 1 check 4 - Day-to-day variability of tasmin = tasmax - dtr

**Ad hoc test, outside the chain**: it reads only the cache of notebook 3 (`outputs/step1_check/cache/subgrid_*.npz`,
sums over every domain pixel, season and group of years) and writes figures / tables in `outputs/` next to this
notebook. Run notebook 3 first.

**Question.** The QDM adjusts tasmax and dtr; tasmin = tasmax - dtr is not adjusted itself. Its variance is

var(tasmin) = var(tasmax) + var(dtr) - 2 corr(tasmax, dtr) sd(tasmax) sd(dtr)

so a wrong correlation between tasmax and dtr (inherited from EC-Earth3: in spring 0.39 in the product against 0.54 in
ERA5, notebook 3) becomes a wrong variance of tasmin, which LISFLOOD receives.

**Method.** Calibration years 1981-2014, all domain pixels, four seasons. Variability = SD of the daily anomaly from
the mean of the same calendar month of the same year (day-to-day variability, as in notebook 3). For each pixel:
the ratio product / ERA5 of the SD of tasmax, dtr and tasmin, the difference of corr(tasmax, dtr), and the part of the
tasmin error explained by that correlation (tasmin SD obtained with the SDs of the product and the correlation of
ERA5). Noise: the same three splits of ERA5 into 17-year halves as notebook 3, t = difference / noise, |t| > `T_CRIT`
significant. Tables for the domain and the PRUDENCE regions, maps.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, glob, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy import stats
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None
cfg = bu.load_config(CONFIG_PATH)
RUN = os.environ.get("STEP1_RUN", "stab2050nohos")
BASE = Path(cfg["data_root"]) / "elisa" / "GCM_downscaling"
INP = BASE / "guido_inputs"
YEARS = (1981, 2014)
SEASONS = ("DJF", "MAM", "JJA", "SON")                       # order of the cache of notebook 3
SPLITS = {"halves": ((0, 1), (2, 3)), "even/odd": ((0, 2), (1, 3)), "mixed": ((0, 3), (1, 2))}   # as notebook 3
T_CRIT = float(stats.t.ppf(0.975, len(SPLITS)))
PRUDENCE = {"BI": ("British Isles", (50, 59), (-10, 2)), "IP": ("Iberian Peninsula", (36, 44), (-10, 3)),
            "FR": ("France", (44, 50), (-5, 5)), "ME": ("Mid-Europe", (48, 55), (2, 16)),
            "SC": ("Scandinavia", (55, 70), (5, 30)), "AL": ("Alps", (44, 48), (5, 15)),
            "MD": ("Mediterranean", (36, 44), (3, 25)), "EA": ("Eastern Europe", (44, 55), (16, 30))}   # lat, lon boxes

CACHE = bu.out_dir(cfg, "step1_check", "cache")
OUT = Path.cwd() / "outputs"; OUT.mkdir(exist_ok=True)
TAG = f"{RUN}_{YEARS[0]}-{YEARS[1]}"


def latest(name):
    fs = sorted(glob.glob(str(CACHE / f"subgrid_{name}_*tasmax-dtr*_{TAG}_*.npz")), key=os.path.getmtime)
    assert fs, f"no cache of notebook 3 for {name} in {CACHE}: run notebook 3 first"
    print(f"{name}: {Path(fs[-1]).name}")
    return dict(np.load(fs[-1]))


ACC = {name: latest(name) for name in ("ERA5", "PROD")}
assert "Saa_tasmin" in ACC["ERA5"] and "Sxy_tasmax_dtr" in ACC["ERA5"], "cache without tasmin / tasmax-dtr"

# domain pixels as in notebook 3 (same ERA5 grid and mask)
def first_field(path, var):
    ds = xr.open_dataset(path, **bu._TIME_KW)
    da = ds[var] if var in ds.data_vars else ds[list(ds.data_vars)[0]]
    da = da.isel({d: 0 for d in da.dims if d not in ("lat", "lon", "latitude", "longitude", "y", "x")}, drop=True)
    return bu.norm_coords(da).transpose("lat", "lon")


F0 = first_field(INP / "era5" / "tasmax_era5_1981_2014_europe.nc", "tasmax")
FLAT, FLON = F0.lat.values, F0.lon.values
NP = ACC["ERA5"]["Saa_tasmax"].shape[-1]
DOM = bu.load_domain_mask(cfg, "era5", like=F0).values.ravel()
P = np.flatnonzero(DOM)
if P.size != NP:                                             # notebook 3 also drops pixels outside the EC-Earth3 grid
    G0 = first_field(INP / RUN / f"tasmax_EC-Earth3-{RUN}_HISTORICAL_europe.nc", "tasmax")
    ye, xe = bu._edges(G0.lat.values), bu._edges(G0.lon.values)
    iy = np.searchsorted(ye, FLAT) - 1; ix = np.searchsorted(xe, FLON) - 1
    inside = (((iy >= 0) & (iy < G0.lat.size))[:, None] & ((ix >= 0) & (ix < G0.lon.size))[None, :]).ravel()
    P = np.flatnonzero(DOM & inside)
assert P.size == NP, f"{P.size} domain pixels here, {NP} in the cache"
PLAT, PLON = FLAT[P // FLON.size], FLON[P % FLON.size]
WGT = np.cos(np.deg2rad(PLAT))
REGIONS = {"DOMAIN": np.ones(NP, bool),
           **{r: (PLAT >= la[0]) & (PLAT <= la[1]) & (PLON >= lo[0]) & (PLON <= lo[1]) for r, (_, la, lo) in PRUDENCE.items()}}
print(f"{NP} domain pixels")

## 1. SD of tasmax, dtr, tasmin and corr(tasmax, dtr) per pixel; part of the tasmin error due to the correlation

In [ ]:
def moments(acc, groups):
    g = list(groups)
    n = acc["n"][g].sum(0)[:, None]
    sd = {v: np.sqrt(acc[f"Saa_{v}"][g].sum(0) / n) for v in ("tasmax", "dtr", "tasmin")}
    with np.errstate(invalid="ignore", divide="ignore"):
        r = acc["Sxy_tasmax_dtr"][g].sum(0) / np.sqrt(acc["Saa_tasmax"][g].sum(0) * acc["Saa_dtr"][g].sum(0))
    return {**{f"sd_{v}": x for v, x in sd.items()}, "corr": r}


M = {name: moments(ACC[name], (0, 1, 2, 3)) for name in ACC}
HALF = {(sp, j): moments(ACC["ERA5"], gg[j]) for sp, gg in SPLITS.items() for j in (0, 1)}
e, p = M["ERA5"], M["PROD"]
# identity: var(tasmin) = var(tasmax) + var(dtr) - 2 corr sd sd (exact with the sums of notebook 3)
rec = np.sqrt(np.maximum(p["sd_tasmax"] ** 2 + p["sd_dtr"] ** 2 - 2 * p["corr"] * p["sd_tasmax"] * p["sd_dtr"], 0))
assert np.nanmax(np.abs(rec - p["sd_tasmin"])) < 1e-3, "identity of the variances not satisfied"
# tasmin SD with the SDs of the product and the correlation of ERA5: what tasmin would be with the right correlation
alt = np.sqrt(np.maximum(p["sd_tasmax"] ** 2 + p["sd_dtr"] ** 2 - 2 * e["corr"] * p["sd_tasmax"] * p["sd_dtr"], 0))


def ratio_m1(a, b):
    with np.errstate(invalid="ignore", divide="ignore"):
        return a / b - 1.0


QUANT = {"sd_tasmin": ("SD of tasmin, product / ERA5 - 1", "rel"),
         "sd_tasmax": ("SD of tasmax, product / ERA5 - 1", "rel"),
         "sd_dtr": ("SD of dtr, product / ERA5 - 1", "rel"),
         "corr": ("corr(tasmax, dtr), product - ERA5", "abs")}


def diff(a, b, kind):
    return ratio_m1(a, b) if kind == "rel" else a - b


PIX = {}
for q, (label, kind) in QUANT.items():
    d = diff(p[q], e[q], kind)
    D = [diff(HALF[sp, 0][q], HALF[sp, 1][q], kind) for sp in SPLITS]
    nz = np.sqrt(sum(x ** 2 for x in D) / len(D) / 2)
    with np.errstate(invalid="ignore", divide="ignore"):
        PIX[q] = {"diff": d, "t": d / nz}
PIX["tasmin_with_era5_corr"] = {"diff": ratio_m1(alt, e["sd_tasmin"])}
print("pixels with a significant tasmin SD error (|t| > %.1f), per season: %s" % (
    T_CRIT, ", ".join(f"{s} {100 * np.nanmean(np.abs(PIX['sd_tasmin']['t'][i]) > T_CRIT):.0f} %" for i, s in enumerate(SEASONS))))

## 2. Domain and PRUDENCE regions

Area-weighted regional values. `tasmin_err` = SD of tasmin product / ERA5 - 1; `tasmin_err_with_era5_corr` = the same
with the correlation of ERA5 (what is left once the correlation is right: the part due to the SDs of tasmax and dtr);
`share_from_corr` = part of the tasmin error due to the correlation.

In [ ]:
def rmean(x, reg):
    ok = REGIONS[reg] & np.isfinite(x)
    return float((WGT[ok] * x[ok]).sum() / WGT[ok].sum()) if ok.any() else np.nan


ROWS = []
for si, s in enumerate(SEASONS):
    for reg in REGIONS:
        row = dict(season=s, region=reg)
        for v in ("tasmax", "dtr", "tasmin"):
            row[f"sd_{v}_ERA5"], row[f"sd_{v}_PROD"] = rmean(e[f"sd_{v}"][si], reg), rmean(p[f"sd_{v}"][si], reg)
        row["corr_ERA5"], row["corr_PROD"] = rmean(e["corr"][si], reg), rmean(p["corr"][si], reg)
        for q in QUANT:
            ERA_H = [rmean(HALF[sp, j][q][si], reg) for sp in SPLITS for j in (0, 1)]
            Dq = [diff(ERA_H[2 * k], ERA_H[2 * k + 1], QUANT[q][1]) for k in range(len(SPLITS))]
            nz = np.sqrt(sum(x ** 2 for x in Dq) / len(Dq) / 2)
            dq = diff(rmean(p[q][si], reg), rmean(e[q][si], reg), QUANT[q][1])
            row[f"{q}_diff"], row[f"{q}_t"] = dq, (dq / nz if nz > 0 else np.nan)
        row["tasmin_err_with_era5_corr"] = ratio_m1(rmean(alt[si], reg), row["sd_tasmin_ERA5"])
        row["share_from_corr"] = (1 - row["tasmin_err_with_era5_corr"] / row["sd_tasmin_diff"]
                                  if abs(row["sd_tasmin_diff"]) > 1e-6 else np.nan)
        ROWS.append(row)
RES = pd.DataFrame(ROWS)
RES.to_csv(OUT / f"tasmin_variability_{TAG}.csv", index=False)
cols = ["season", "region", "sd_tasmin_ERA5", "sd_tasmin_PROD", "sd_tasmin_diff", "sd_tasmin_t", "sd_tasmax_diff",
        "sd_dtr_diff", "corr_ERA5", "corr_PROD", "tasmin_err_with_era5_corr", "share_from_corr"]
with pd.option_context("display.width", 250, "display.max_columns", None, "display.float_format", "{:.3f}".format,
                       "display.max_rows", 100):
    display(RES[cols])
print("saved:", OUT / f"tasmin_variability_{TAG}.csv")

## 3. Maps

One row per season: SD of tasmin, of tasmax and of dtr (product / ERA5 - 1), corr(tasmax, dtr) product - ERA5, the
tasmin error with the correlation of ERA5, and t of the tasmin error. PRUDENCE boxes on the first column.

In [ ]:
BORDERS = bu.borders(cfg, "10M")
EXTENT = (PLON.min() - 0.5, PLON.max() + 0.5, PLAT.min() - 0.5, PLAT.max() + 0.5)


def to_grid(x):
    g = np.full(FLAT.size * FLON.size, np.nan); g[P] = x
    return g.reshape(FLAT.size, FLON.size)


def draw(ax, x, title, vmin, vmax, boxes=False):
    pm = ax.pcolormesh(FLON, FLAT, to_grid(x), shading="nearest", cmap="RdBu_r", vmin=vmin, vmax=vmax)
    bu.draw_borders(ax, BORDERS, lw=0.3)
    if boxes:
        for r, (_, la, lo) in PRUDENCE.items():
            ax.add_patch(Rectangle((lo[0], la[0]), lo[1] - lo[0], la[1] - la[0], fill=False, lw=0.6, ec="k"))
            ax.text(lo[0] + 0.3, la[1] - 1.5, r, fontsize=6, clip_on=True)
    ax.set_xlim(EXTENT[:2]); ax.set_ylim(EXTENT[2:]); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=8)
    plt.colorbar(pm, ax=ax, shrink=0.75)


PANELS = [("sd_tasmin", "SD tasmin product / ERA5 - 1", 0.3), ("sd_tasmax", "SD tasmax product / ERA5 - 1", 0.3),
          ("sd_dtr", "SD dtr product / ERA5 - 1", 0.3), ("corr", "corr(tasmax, dtr) product - ERA5", 0.4),
          ("tasmin_with_era5_corr", "SD tasmin / ERA5 - 1 with the ERA5 correlation", 0.3)]
fig, axes = plt.subplots(len(SEASONS), len(PANELS) + 1, figsize=(4.2 * (len(PANELS) + 1), 3.3 * len(SEASONS)), squeeze=False)
for si, s in enumerate(SEASONS):
    for j, (q, title, m) in enumerate(PANELS):
        draw(axes[si, j], PIX[q]["diff"][si], f"{s} {title}", -m, m, boxes=(j == 0))
    t = PIX["sd_tasmin"]["t"][si]
    draw(axes[si, -1], t, f"{s} t of the tasmin SD error ({100 * np.nanmean(np.abs(t) > T_CRIT):.0f} % |t| > {T_CRIT:.1f})", -6, 6)
plt.tight_layout(); plt.savefig(OUT / f"map_tasmin_variability_{TAG}.png", dpi=90); plt.show(); plt.close(fig)